# NB11 — SHAP Explicabilidad sobre M2
**ZMM Movilidad Predictiva**

Genera summary plot, dependence plots y waterfall para interpretar el modelo de clasificación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

RUTA_PROCESSED = '../data_processed/'
RUTA_OUTPUTS   = '../outputs/'

print('='*65)
print('NB11: SHAP EXPLICABILIDAD SOBRE M2')
print('='*65)

In [ ]:
# Load model and data
with open(RUTA_OUTPUTS + 'modelo_m2_rf.pkl', 'rb') as f:
    m2 = pickle.load(f)
model, features, nombre = m2['modelo'], m2['features'], m2['nombre']
print(f'Modelo: {nombre}')

df = pd.read_csv(RUTA_PROCESSED + 'super_tabla_con_clusters.csv', parse_dates=['fecha_hora'])
df = df[(df['fecha_hora'] >= '2023-01-01') & (df['fecha_hora'] <= '2025-12-31 23:00:00')].copy()

# Recreate features
for sf in ['dist_industrial_promedio','dist_industrial_minima','siniestros_en_zona','masa_laboral_max']:
    df[sf] = df[sf].fillna(df[sf].median())
df['tiene_dato_espacial'] = 1  # all filled now
if 'tipo_dia' in df.columns:
    df = pd.concat([df, pd.get_dummies(df['tipo_dia'], prefix='td', drop_first=True)], axis=1)

df['nivel_riesgo'] = pd.cut(df['siniestros_zona_industrial'], bins=[-1,2,5,100], labels=[0,1,2]).astype(int)

X = df[features].copy(); y = df['nivel_riesgo']
np.random.seed(42)
idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
X_s, y_s = X.iloc[idx], y.iloc[idx]
print(f'Sample: {len(X_s)}')

In [ ]:
# SHAP
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_s)
sa = np.array(sv)
if isinstance(sv, list): sl = sv
elif sa.ndim == 3 and sa.shape[2] == 3: sl = [sa[:,:,i] for i in range(3)]
elif sa.ndim == 3 and sa.shape[0] == 3: sl = [sa[i] for i in range(3)]
else: sl = [sv]
shap_alto = sl[2]
print(f'SHAP shape: {shap_alto.shape}')

In [ ]:
# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_alto, X_s, feature_names=features, show=False)
plt.title('SHAP Summary — Impacto en Riesgo ALTO')
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dependence: hora_del_dia
fig, ax = plt.subplots(figsize=(10, 6))
shap.dependence_plot('hora_del_dia', shap_alto, X_s, feature_names=features, show=False, ax=ax)
plt.title('SHAP Dependence: Hora del Dia')
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'shap_dependence_hora.png', dpi=150, bbox_inches='tight')
plt.show()

# Dependence: intensidad_hora_pico
fig, ax = plt.subplots(figsize=(10, 6))
shap.dependence_plot('intensidad_hora_pico', shap_alto, X_s, feature_names=features, show=False, ax=ax)
plt.title('SHAP Dependence: Intensidad Hora Pico')
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'shap_dependence_pico.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall: caso alto riesgo
idx_alto = np.where(y_s.values == 2)[0]
if len(idx_alto) > 0:
    ej = idx_alto[0]
    sv_ej = explainer.shap_values(X_s.iloc[[ej]])
    sa_ej = np.array(sv_ej)
    if isinstance(sv_ej, list): sv2 = sv_ej[2][0]; ev = explainer.expected_value[2]
    elif sa_ej.ndim == 3 and sa_ej.shape[2] == 3: sv2 = sa_ej[0,:,2]; ev = explainer.expected_value[2]
    elif sa_ej.ndim == 3 and sa_ej.shape[0] == 3: sv2 = sa_ej[2][0]; ev = explainer.expected_value[2]
    else: sv2 = sa_ej[0]; ev = explainer.expected_value

    plt.figure(figsize=(12, 8))
    shap.waterfall_plot(shap.Explanation(values=sv2, base_values=ev,
        data=X_s.iloc[ej], feature_names=features), show=False)
    plt.title('Waterfall: Caso de Alto Riesgo')
    plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'shap_waterfall_alto_riesgo.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Feature ranking SHAP vs RF
mean_shap = np.mean([np.abs(s).mean(axis=0) for s in sl], axis=0)
shap_imp = pd.DataFrame({'feature': features, 'mean_abs_shap': mean_shap}).sort_values('mean_abs_shap', ascending=False)
print('TOP 10 FEATURES POR SHAP:')
for _, r in shap_imp.head(10).iterrows():
    print(f'  {r.feature}: {r.mean_abs_shap:.4f}')

print(f'\nRF top 3:   {m2["feature_importance"].head(3)["feature"].tolist()}')
print(f'SHAP top 3: {shap_imp.head(3)["feature"].tolist()}')
print('\nNB11 COMPLETADO')